# Method 1 (Appendix E): un-truncated best fit up to 7 MeV

Fit of the integrated flux $\Phi(E_\nu)-\Phi(7\,\mathrm{MeV})$ **without** truncating the
response functions at 2 MeV: the flux above 2 MeV is part of the signal (no $h_i$ background),
no external background, same Neyman $\chi^2$, non-negativity and monotonicity constraints,
and vertex selection, as in the paper.

**Grid (Graciela's prescription)** — intervals chosen so that all $\mathcal{R}_{ij}$ have the
same size: uniform below 2 MeV (matching the paper), adaptive above 2 MeV. Note the kernels
*decrease* with $E_\nu$ above 2 MeV (recoils spread beyond the 120 eV window,
$E_R^{\max}(2\,\mathrm{MeV})\simeq120$ eV), so the adaptive intervals get *wider* with energy.

**Success criterion** (Graciela, Sep 21): a piecewise-constant fit at all energies with at most
$d-1$ downward steps. The raw vertex has $d$ steps; one $\Delta\chi^2=0$ merge gives exactly
$d-1$ (30 for 1 eV, 28 for 5 eV), saturating the FDS bound.

**Inputs** — `data/curlyR_knots_<thr>.csv`: box-smeared kernels on the official 0.01 MeV knots,
regenerated with the repository's Mathematica pipeline (read-only) via
```
/Applications/Wolfram.app/Contents/MacOS/wolframscript -file driver_kernels.wls <1eV|5eV>
```
Below 2 MeV the *published* response-matrix columns (`CRmat{NB}_originalUnit.csv`) are used
verbatim; the regenerated kernels only fill the columns above 2 MeV.

In [ ]:
# ---- parameters ----------------------------------------------------
THR = '1eV'    # '1eV' or '5eV'
NB  = 180      # uniform intervals below 2 MeV (matches the paper)
REPO = '/Users/koichiro/Documents/Claude/neutrinoAnalysis_v2'

In [ ]:
import os, sys, json
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import LogLocator
from scipy import integrate
import cvxpy as cp

EMIN = {'1eV': 0.18, '5eV': 0.41}[THR]
HERE = os.path.join(REPO, 'notes', 'method1')

kn = pd.read_csv(os.path.join(HERE, 'data', f'curlyR_knots_{THR}.csv'))
Ek = kn.iloc[:, 0].values            # MeV, 0.01 spacing
K  = kn.iloc[:, 1:].values.T         # (m bins, npts), cm^2/(MeV kg)
m  = K.shape[0]
print(f'kernel knots: {K.shape[1]} pts x {m} bins, E = {Ek[0]}..{Ek[-1]} MeV')

def pl_integral(y, a, b):
    xs = np.r_[a, Ek[(Ek > a) & (Ek < b)], b]
    ys = np.interp(xs, Ek, y)
    return np.sum(0.5*(ys[1:]+ys[:-1])*np.diff(xs))

def build_matrix(edges):
    M = np.zeros((m, len(edges)-1))
    for j in range(len(edges)-1):
        for i in range(m):
            M[i, j] = pl_integral(K[i], edges[j], edges[j+1])
    return M

## Response matrix: validation and splice
Row sums of the regenerated kernels agree with the published matrix (scale = 1); the fine-scale
shape differs from the legacy kernels, so below 2 MeV the published columns are used verbatim.

In [ ]:
edgesLow = np.linspace(EMIN, 2.0, NB+1)
Mlow = build_matrix(edgesLow)
cand = [os.path.join(REPO, THR, 'CRmat', 'originalUnit', f'CRmat{NB}_originalUnit.csv'),
        os.path.join(REPO, 'Mathematica', THR, 'output', f'CRmat{NB}_originalUnit.csv')]
ref = np.genfromtxt(next(f for f in cand if os.path.exists(f)), delimiter=',')
rs = Mlow.sum(1)/ref.sum(1)
s = np.median(rs)
print(f'scale (row sums) = {s:.6f},  max row deviation = {np.abs(rs/s-1).max():.2e}')

## Adaptive grid above 2 MeV
Each interval above 2 MeV integrates $\sum_i \mathcal{R}_i$ to the same value $T$ as the last
uniform interval below 2 MeV.

In [ ]:
S = K.sum(axis=0)
T = pl_integral(S, edgesLow[-2], edgesLow[-1])
hi = [2.0]
while hi[-1] < 7.0 - 1e-12:
    a0 = hi[-1]
    if pl_integral(S, a0, 7.0) <= T:
        hi.append(7.0); break
    lo_, hi_ = a0, 7.0
    for _ in range(60):
        mid = 0.5*(lo_+hi_)
        (lo_, hi_) = (mid, hi_) if pl_integral(S, a0, mid) < T else (lo_, mid)
    hi.append(0.5*(lo_+hi_))
hi = np.array(hi)
edges = np.r_[edgesLow, hi[1:]]
n = len(edges)-1
print(f'{NB} uniform + {len(hi)-1} adaptive -> n = {n};  '
      f'adaptive widths {np.diff(hi).min():.4f}..{np.diff(hi).max():.4f} MeV')

M1 = build_matrix(edges)/s
M1[:, :NB] = ref            # published columns verbatim below 2 MeV

## Fit
The analysis class is reused with instance-level overrides: data = full rate (`Ratebin7`),
nothing subtracted, matrix extended to 7 MeV.

In [ ]:
sys.path.insert(0, os.path.join(REPO, THR)); os.chdir(os.path.join(REPO, THR))
import neutrino_analysis_band as nab

def make_analysis(solver='osqp'):
    a = nab.NeutrinoAnalysis(background_scenario='none', intervals='180',
                             GeV=0.32e16, solver=solver, T=3.0)
    conv_mat = a.cm**2/(10**3*a.gram)*(10**3*a.gram)*a.yr
    a.CRmat = M1*conv_mat
    a.n = n
    a.M_matrix = a.c*a.CRmat
    a.data_vector = a.c*a.Ratebin7      # >2 MeV neutrinos are signal; no h_i subtraction
    a.Bkg_vector = np.zeros(a.m)
    a._build_ordering_constraint()
    a._dmb_default, a._inv_d_default = a._make_dmb_inv(a.data_vector)
    a._hess_default = a._build_hessian_from(a._dmb_default, a._inv_d_default)
    a._baseline_result = None
    a.set_solver(solver)
    if a._backend.name == 'osqp':
        a._backend.vertex_select = True
    return a

a = make_analysis()
res = a.optimize(a.data_vector)
conv = a.cm**2*a.sec
x_vertex = res.x.copy()
chi0 = res.fun/a.c
print(f'vertex fit: chi2/c = {chi0:.2e},  x(Emin) = {x_vertex[0]*conv:.4e} cm^-2 s^-1')

In [ ]:
# stability: vertex off + random restarts give the identical solution
av = make_analysis(); av._backend.vertex_select = False
rv = av.optimize(av.data_vector)
print(f'vertex off : chi2/c = {rv.fun/a.c:.2e}')
rng = np.random.default_rng(1)
for k in range(3):
    x0 = np.sort(rng.uniform(0, 2*x_vertex[0], n))[::-1]
    rk = a.optimize(a.data_vector, x0=x0)
    print(f'restart {k}: chi2/c = {rk.fun/a.c:.2e},  '
          f'max|dx|/x0 = {np.max(np.abs(rk.x-x_vertex))/x_vertex[0]:.1e}')

## Exact merge to $d-1$ downward steps
Greedy merging of adjacent treads, keeping $M x = \mu$ *exactly* ($\Delta\chi^2 = 0$).
Both thresholds land exactly on the FDS bound $d-1$.

In [ ]:
Ms = a.M_matrix/a.c
mu = Ms @ x_vertex
tol = 1e-6*x_vertex[0]

def treads(x):
    b = [[0, 0]]
    for j in range(1, n):
        if abs(x[j]-x[j-1]) < tol: b[-1][1] = j
        else: b.append([j, j])
    return [tuple(t) for t in b]

XS = float(x_vertex[0])
def feasible(bl):
    y = cp.Variable(n, nonneg=True)
    cons = [(Ms*XS)@y == mu, y[:-1] >= y[1:]]
    for s0, e0 in bl:
        if e0 > s0: cons.append(y[s0:e0] == y[s0+1:e0+1])
    p = cp.Problem(cp.Minimize(0), cons)
    try:
        p.solve(solver=cp.HIGHS, time_limit=15.0)
        return p.status in ('optimal', 'optimal_inaccurate')
    except Exception:
        return False

blocks = treads(x_vertex)
improved = True
while improved:
    improved = False
    order = sorted(range(len(blocks)-1),
                   key=lambda k: min(blocks[k][1]-blocks[k][0], blocks[k+1][1]-blocks[k+1][0]))
    for k in order:
        trial = blocks[:k] + [(blocks[k][0], blocks[k+1][1])] + blocks[k+2:]
        if feasible(trial):
            blocks = trial; improved = True; break

y = cp.Variable(n, nonneg=True)
cons = [(Ms*XS)@y == mu, y[:-1] >= y[1:]]
for s0, e0 in blocks:
    if e0 > s0: cons.append(y[s0:e0] == y[s0+1:e0+1])
tw = 1000.0**(np.arange(n)/max(n-1, 1))
cp.Problem(cp.Minimize(tw@y), cons).solve(solver=cp.HIGHS, time_limit=30.0)
x_fit = y.value*XS
chi_f = float(np.sum((a.data_vector - a.M_matrix@x_fit)**2
              / np.where(a.data_vector > 0, a.data_vector, 1)))/a.c
d = a.m
print(f'd = {d}:  vertex steps {len(treads(x_vertex))-1}  ->  merged steps {len(blocks)-1}'
      f'  (d-1 = {d-1});  chi2/c = {chi_f:.2e}')

## Figure (paper band-comparison style)

In [ ]:
xg = np.logspace(-2, np.log10(7.0), 4000)
ys = np.interp(xg, a.fig1Solid['MeV'], a.fig1Solid['cm**-2sec-1MeV-1'])
yd = np.interp(xg, a.fig1dashed['MeV'], a.fig1dashed['cm**-2sec-1MeV-1'])
Phi_s = np.array([integrate.trapezoid(ys[i:], xg[i:]) for i in range(len(xg))])
Phi_d = np.array([integrate.trapezoid(yd[i:], xg[i:]) for i in range(len(xg))])

norm = 1e12
plt.style.use(os.path.join(REPO, '1eV', 'physrev.mplstyle'))
plt.figure(figsize=(8, 6))
plt.plot(xg, Phi_s/norm, color='black', lw=3, label='With NC')
plt.plot(xg, Phi_d/norm, color='black', lw=3, ls='dashed', label='Without NC')
plt.scatter(edges[:-1], x_fit*conv/norm, s=10, marker='o', color='#0072B2',
            zorder=5, label='No Bkg Best-fit')
plt.xscale('log'); plt.xlim(1e-1, 7.3)
plt.xlabel(r"$E_\nu$ [MeV]", fontsize=30)
plt.ylabel(nab._phi_ylabel(norm).replace('<~2~', '<~7~'), fontsize=30)
plt.gca().xaxis.set_minor_locator(LogLocator(base=10.0, subs=np.arange(1., 10)*0.1, numticks=20))
plt.tick_params(axis='both', which='both', labelsize=23)
plt.legend(loc='upper right', fontsize=15, frameon=False)
plt.tight_layout()
plt.show()